<a href="https://colab.research.google.com/github/ochilovu2010/IOAI/blob/main/NEOAI/BROKEN_BERT_BASELINE_SOLUTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

neoai_2025_broken_bert_path = kagglehub.competition_download('neoai-2025-broken-bert')

print('Data source import complete.')


# TASK DESCRIPTION

**Legend:**

Young Alex has a beloved BERT model that he carries everywhere on his trusty flash drive. One day, during an excursion along the River Styx, a few drops of water landed on the precious device, corrupting the model's weights.

Heartbroken, Alex rushed home to fix the neural network. After quick analysis, he discovered only the token embeddings were damaged - the rest of the architecture (attention blocks and heads) remained perfectly intact. Now he needs to restore the model's performance on Sentiment Analysis Task.

**Task:**

You need to fix the broken vectors of the Embeddings matrix of the model so as to improve the quality of the model on the task of text sentiment analysis.

**Restrictions:**

- You can not use any other transformer based pre-trained models and LLMs.

- You can not any additional data

- You can not fine-tune or pre-train model

===

When you make a submit, make a Quick Save of the notebook, otherwise we may reject your solution.

You must solve this task on KAGGLE (YOU CAN'T USE CLOUD.RU)

==========

**Легенда:**

Young Alex имеет любимую модель BERT, которую он везде носит на своей надежной флешке. Однажды, во время экскурсии вдоль реки Стикс, несколько капель воды попало на драгоценное устройство, повредив веса модели.

С разбитым сердцем Алекс поспешил домой, чтобы починить нейронную сеть. После быстрого анализа он обнаружил, что повреждены только эмбеддинги токенов — остальная архитектура (блоки внимания и головы) осталась полностью нетронутой.

Теперь ему нужно восстановить производительность модели, оставив все остальные веса замороженными (никакие изменения в механизмах внимания или других компонентах не допускаются). Ваша задача — помочь Алексу достичь этой цели, не нарушив его ностальгическую привязанность к оригинальной модели.

**Задача:**

Вам необходимо починить сломанные вектора матрицы Embeddings модели так, чтобы улучшить качество модели на задаче анализа тональности текста.

**Ограничения:**

- Вы не можете использовать никакие другие предобученные модели на основе архитектуры Трансформер и LLM.

- Вы не можете использовать никакие дополнительные данные.

- Вы не можете дообучать или предобучать модель.

===

При отправке решения сделайте Quick Save ноутбука, иначе мы можем отклонить ваше решение.

Эту задачу необходимо решить на KAGGLE (ВЫ НЕ МОЖЕТЕ ИСПОЛЬЗОВАТЬ CLOUD.RU)


# DEPENDINGS

In [ ]:
import numpy as np
import pandas as pd
import torch
np.random.seed(21)

# LOAD DATASET

In [ ]:
val_data_path = "/kaggle/input/neoai-2025-broken-bert/val_dataset.csv"
test_data_path = "/kaggle/input/neoai-2025-broken-bert/test.csv"

val_df = pd.read_csv(val_data_path)

test_df = pd.read_csv(test_data_path)

# LOAD TOKENIZER & MODEL

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("Ilseyar-kfu/broken_bert")
tokenizer = AutoTokenizer.from_pretrained("Ilseyar-kfu/broken_bert")

In [ ]:
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
val_encodings = tokenizer(val_df["text"].to_list(), truncation=True, padding=True, max_length=256)
val_dataset = Dataset(val_encodings, val_df["labels"].to_list())

In [ ]:
texts_2_score = val_df["text"].to_list() + test_df["text"].to_list()

In [ ]:
embeddings = model.bert.embeddings.word_embeddings.weight.detach()
print(embeddings.shape)

torch.Size([30522, 768])


In [ ]:
model_embeddings = model.bert.embeddings.word_embeddings.weight.detach().cpu()
zero_rows = (model_embeddings == 0).all(dim=1)
zero_indices = torch.nonzero(zero_rows).squeeze().numpy()

In [ ]:
print(len(zero_indices), len(model_embeddings))

12208 30522


In [ ]:
token_to_ids = tokenizer.get_vocab()
ids_to_token = {v: k for k,v in token_to_ids.items()}
non_zero_ids_to_token = {v: k for k,v in token_to_ids.items() if v not in indexes}

In [ ]:
def get_sub_tokens(token, ids_to_token):
  sub_tokens = []
  for idx, sub_token in ids_to_token.items():
    if sub_token in token:
      sub_tokens.append(idx)
  return sub_tokens

In [ ]:
from tqdm import tqdm
for zero_index in tqdm(zero_indices):
  zero_token = ids_to_token[zero_index]
  sub_tokens = get_sub_tokens(zero_token, non_zero_ids_to_token)
  if len(sub_tokens) != 0:
    mean_embedding = model_embeddings[sub_tokens].mean(dim=0)
    embeddings[zero_index] = mean_embedding
  else:
    embeddings[zero_index] = torch.rand(1, model_embeddings.shape[1])

100%|██████████| 12208/12208 [00:21<00:00, 576.45it/s]


In [ ]:
print(missing_embeddings.shape, len(indexes))

torch.Size([12208, 768]) 0


In [ ]:
print(sum((embeddings == 0).all(dim = 1)))


tensor(499)


# MODEL CHANGES

In [ ]:

# There's magic going on here!!! And we get very new !!! new_embedings !!!
new_embedings = embeddings.cpu()

model.bert.embeddings.word_embeddings.weight = torch.nn.Parameter(torch.Tensor(new_embedings))

# EVALUATION

In [ ]:
from sklearn.metrics import f1_score
from numpy import argmax
from transformers import pipeline
import wandb
wandb.init(mode= "disabled")

In [ ]:
from sklearn.metrics import classification_report

def evaluate_on_validation(model, tokenizer, df_val):
    label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    answ = classifier.predict(list(df_val["text"]))
    answ = [label_2_dict[el["label"]] for el in answ]

    # print(f1_score(p.label_ids, preds, average='macro'))
    print(classification_report(df_val["labels"], answ))

In [ ]:
evaluate_on_validation(model, tokenizer, val_df)

Device set to use cuda:0


              precision    recall  f1-score   support

    negative       0.61      0.30      0.40       935
     neutral       0.35      0.80      0.48       759
    positive       0.64      0.23      0.34       806

    accuracy                           0.43      2500
   macro avg       0.53      0.44      0.41      2500
weighted avg       0.54      0.43      0.41      2500



# MODEL SCORING
When you make a submit,
1. Make a Quick Save of the notebook, otherwise we may reject your solution!
2. Add notebook version to the comment for the submit.

===

При отправке решения:

1. Сделайте Quick Save ноутбука, иначе мы можем отклонить ваше решение!
2. Добавьте версию ноутбука в комментарий к отправке.

In [ ]:
import hashlib

def create_submission(model, tokenizer, df_test):
    label_2_dict = {'LABEL_0': 'neutral', "LABEL_1" : 'positive', "LABEL_2": 'negative'}
    classifier = pipeline("text-classification", model= model, tokenizer = tokenizer)
    answ = classifier.predict(list(df_test["text"]))
    answ = [label_2_dict[el["label"]] for el in answ]

    df = pd.DataFrame({"labels" : answ, "id": df_test['id']})
    hsh = hashlib.sha256(df.to_csv(index=False).encode('utf-8')).hexdigest()[:8]
    submit_path = f"submit_{hsh}.csv"
    print(f"SUBMIT_NAME: {submit_path}")
    print(df.head(10))
    df.to_csv(submit_path,index=False)

In [ ]:
create_submission(model, tokenizer, test_df)

Device set to use cuda:0


SUBMIT_NAME: submit_7377155d.csv
     labels    id
0  positive  5000
1   neutral  5001
2   neutral  5002
3   neutral  5003
4  positive  5004
5  negative  5005
6   neutral  5006
7   neutral  5007
8  negative  5008
9   neutral  5009


In [ ]:
model.bert.embeddings.word_embeddings.weight.detach()

tensor([[-0.0102, -0.0615, -0.0265,  ..., -0.0198, -0.0372, -0.0097],
        [-0.3350,  0.7386,  0.9621,  ..., -0.3835,  1.2141,  1.1882],
        [-0.0197, -0.0627, -0.0326,  ..., -0.0165, -0.0420, -0.0032],
        ...,
        [-0.0218, -0.0556, -0.0135,  ..., -0.0043, -0.0151, -0.0249],
        [-0.0462, -0.0565, -0.0019,  ...,  0.0157, -0.0139, -0.0094],
        [ 0.8878,  1.0755,  0.8266,  ...,  0.0944, -0.4929, -0.6805]],
       device='cuda:0')